# macOS Python Environment Setup for Data Fusion
This notebook automates the creation and setup of a Python virtual environment in the `.env/` directory of the project using standard macOS terminal commands.

### Specs of Target System:
* **OS**: macOS
* **Python Version**: 3.8+
* **Target Environment**: `.env/` folder in the project root

---
### Setup Highlights:
1. **PyTorch with MPS/Metal**: Installs standard PyTorch which automatically leverages Apple Silicon GPU acceleration (Metal Performance Shaders).
2. **TensorFlow & tensorflow-metal**: Installs standard TensorFlow and optionally `tensorflow-metal` plugin for Apple Silicon hardware acceleration.
3. **Local editable install**: Installs `data_fusion_project` in editable mode with PowerPoint control features.

---
### How to Run This Notebook:
1. Open this notebook in **VS Code** or **Cursor**.
2. Select a default/system Python interpreter (Python 3.8+) in the top-right corner to run these setup cells.
3. Run the cells sequentially.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

def run_command(command, shell=True):
    "\"\"Runs a system command and streams output to the notebook cell in real-time.\"\""
    print(f"--- Running: {command} ---\n")
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, shell=shell, text=True)
    
    # Stream output line by line
    for line in process.stdout:
        print(line, end="")
    
    process.wait()
    if process.returncode != 0:
        print(f"\n[ERROR] Command failed with exit code: {process.returncode}")
        return False
    else:
        print(f"\n[SUCCESS] Completed successfully.")
        return True

# Resolve project root dynamically
current_dir = Path(os.getcwd()).resolve()
project_root = current_dir
while project_root != project_root.parent:
    if (project_root / "pyproject.toml").exists():
        break
    project_root = project_root.parent

print(f"[INFO] Project root directory: {project_root}")
env_dir = project_root / ".env"
print(f"[INFO] Target environment directory: {env_dir}")

## Step 1: Create the Virtual Environment
We will use standard Python `venv` to create a virtual environment inside the project's `.env/` directory.

In [ ]:
# Run the venv creation command
import sys
python_executable = sys.executable
print(f"Using system python: {python_executable}")
run_command(f'"{python_executable}" -m venv "{env_dir}"') 

## Step 2: Locate Env Executables and Upgrade Pip
We locate `pip` and `python` inside `.env/bin` and upgrade pip.

In [ ]:
env_pip = env_dir / "bin" / "pip"
env_python = env_dir / "bin" / "python"

print(f"Env Pip: {env_pip}")
print(f"Env Python: {env_python}")

# Upgrade pip first
run_command(f'"{env_python}" -m pip install --upgrade pip')

## Step 3: Install macOS PyTorch
Installs standard PyTorch.

In [ ]:
run_command(f'"{env_pip}" install torch torchvision torchaudio')

## Step 4: Install Scientific, ML Stack, and Development Tools

In [ ]:
dependencies = [
    "numpy",
    "pandas",
    "scipy",
    "scikit-learn",
    "matplotlib",
    "pyserial",
    "tqdm",
    "joblib",
    "ipykernel",
    "jupyterlab",
    "notebook",
    "pyyaml",
    "h5py",
    "pyarrow",
    "filterpy",
    "tensorboard",
    "opencv-python",
    "mediapipe",
    "pynput"
]

dep_str = " ".join(dependencies)
run_command(f'"{env_pip}" install {dep_str}')

## Step 5: Install TensorFlow & Apple Silicon Acceleration
Installs standard TensorFlow, and optional `tensorflow-metal` for Apple Silicon GPU acceleration.

In [ ]:
import platform
run_command(f'"{env_pip}" install tensorflow')
if platform.system() == "Darwin" and platform.machine() == "arm64":
    print("[INFO] Apple Silicon detected. Installing tensorflow-metal...")
    run_command(f'"{env_pip}" install tensorflow-metal')
else:
    print("[INFO] Non-ARM macOS or platform. Skipping tensorflow-metal.")

## Step 6: Install Project in Editable Mode
Installs the local `data_fusion_project` package with PowerPoint control utilities.

In [ ]:
run_command(f'"{env_pip}" install -e "{project_root}[control]"')

## Step 7: Register Jupyter Kernel
This makes the environment selectable as a Jupyter kernel called `data_fusion_env_1`.

In [ ]:
run_command(f'"{env_python}" -m ipykernel install --user --name data_fusion_env_1 --display-name "Python (data_fusion_env_1)"') 

## Step 8: Environment Verification

In [ ]:
verification_code = "\nimport sys\nprint(\'=== ENV VERIFICATION ===\')\nprint(\'Python Path:\', sys.executable)\nprint(\'Python Version:\', sys.version)\n\ntry:\n    import numpy as np\n    print(\'NumPy:\', np.__version__)\nexcept Exception as e:\n    print(\'NumPy failed:\', e)\n\ntry:\n    import torch\n    print(\'PyTorch Version:\', torch.__version__)\n    print(\'MPS available in PyTorch:\', torch.backends.mps.is_available())\nexcept Exception as e:\n    print(\'PyTorch failed:\', e)\n\ntry:\n    import tensorflow as tf\n    print(\'TensorFlow Version:\', tf.__version__)\nexcept Exception as e:\n    print(\'TensorFlow failed:\', e)\n\ntry:\n    import cv2\n    print(\'OpenCV:\', cv2.__version__)\nexcept Exception as e:\n    print(\'OpenCV failed:\', e)\n\ntry:\n    import mediapipe as mp\n    print(\'MediaPipe:\', mp.__version__)\nexcept Exception as e:\n    print(\'MediaPipe failed:\', e)\n"

import tempfile
with tempfile.NamedTemporaryFile(suffix=".py", delete=False) as temp_file:
    temp_file.write(verification_code.encode("utf-8"))
    temp_file_name = temp_file.name

try:
    run_command(f'"{env_python}" "{temp_file_name}"')
finally:
    if os.path.exists(temp_file_name):
        os.remove(temp_file_name)
